# Day 3: OOP and OpenAI Codex Walkthrough
**Classes, inheritance, exceptions and dependency hygiene, then structured coding with OpenAI Codex**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
import sys, platform
print("Python", platform.python_version(), "| Colab:", "google.colab" in sys.modules)

# ===== Option B: VS Code / local =====
# Work inside a virtual environment (see section 7):
#   python -m venv .venv  ->  source .venv/bin/activate  (Windows: .venv\Scripts\activate)
#   pip install -r requirements.txt

# Session 1: Classes and objects
## 2. Your first class

In [ ]:
class BankAccount:
    bank_name = "Sakura Bank"          # class attribute, shared

    def __init__(self, owner, balance=0):
        self.owner = owner              # instance attributes
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount
        return self.balance

    def __str__(self):
        return f"{self.owner}: {self.balance}"

a = BankAccount("Hana", 500)
b = BankAccount("Ravi")
a.deposit(250)
print(a, "|", b, "|", BankAccount.bank_name)

## 3. Inheritance, overriding and polymorphism

In [ ]:
class Account:
    def __init__(self, owner):
        self.owner = owner
    def fee(self):
        return 100

class Premium(Account):
    def fee(self):
        return super().fee() // 2

class Student(Account):
    def fee(self):
        return 0

for acc in [Account("A"), Premium("B"), Student("C")]:
    print(f"{type(acc).__name__:<8} fee = {acc.fee()}")

## 4. Properties and encapsulation

In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price          # goes through the setter below

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError("Price cannot be negative")
        self._price = value

    @property
    def price_with_tax(self):
        return round(self._price * 1.18, 2)

p = Product("Mouse", 500)
print(p.price, p.price_with_tax)
try:
    p.price = -10
except ValueError as e:
    print("Blocked:", e)

### TODO 1: Add a `Corporate(Account)` subclass with a fee of 300 and add it to the loop above.

In [ ]:
# TODO: write your code here

# Session 2: Exceptions, modules and dependencies
## 5. Exception handling

In [ ]:
def to_int(text):
    try:
        value = int(text)
    except ValueError:
        print(f"'{text}' is not a number")
        return None
    else:
        print("Converted OK")
        return value
    finally:
        print("Finished checking", repr(text))

print(to_int("42"))
print(to_int("abc"))

In [ ]:
class InsufficientFunds(Exception):
    pass

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFunds(f"Balance {balance}, requested {amount}")
    return balance - amount

try:
    withdraw(100, 500)
except InsufficientFunds as e:
    print("Error:", e)

### TODO 2: Create a custom exception `NegativeAmount` and raise it from `withdraw` when `amount < 0`.

In [ ]:
# TODO: write your code here

## 6. Modules
We write a small module file, then import it.

In [ ]:
%%writefile pricing.py
GST = 0.18

def with_tax(amount):
    return round(amount * (1 + GST), 2)

if __name__ == "__main__":
    print("Running pricing.py directly")

In [ ]:
import pricing
from math import sqrt
print(pricing.with_tax(100), pricing.GST, sqrt(49))

## 7. NEW (v4): Dependency hygiene
**Virtual environment** (run in a VS Code terminal, not in Colab):
```
python -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate
pip install requests==2.32.3
pip freeze > requirements.txt
```
Commit `requirements.txt`. Add `.venv/` and `.env` to `.gitignore`.

Below: a small checker for unpinned requirements, then `pip-audit` (needs internet).

In [ ]:
requirements = """\
fastapi==0.115.0
pydantic==2.9.2
requests
numpy>=1.26
# comments are ignored
"""

def check_pinned(lines):
    """Return requirement lines that are not pinned with ==."""
    out = []
    for line in lines:
        line = line.strip()
        if line and not line.startswith("#") and "==" not in line:
            out.append(line)
    return out

with open("requirements.txt", "w") as f:
    f.write(requirements)
print("Not pinned:", check_pinned(requirements.splitlines()))

In [ ]:
# ===== Option A: Google Colab (needs internet) =====
!pip install -q pip-audit
!pip-audit -r requirements.txt --progress-spinner off || echo "Fix the findings above, or pin the missing versions first"

# ===== Option B: VS Code terminal, inside your venv =====
# pip install pip-audit
# pip-audit -r requirements.txt

# Session 3: NEW (v4): OpenAI Codex in VS Code
Codex runs in VS Code, not in Colab. Follow these steps on your laptop (details in `docs/CODEX_GUIDE.md`):
1. Install VS Code, Python 3.12 or newer and Git.
2. Extensions: search for **OpenAI Codex** and install it.
3. Sign in with the program's **API key** (paste it only into the sign-in box).
4. Select the approved default model (`docs/MODEL_GUIDE.md`). Keep the sandbox on workspace-write with approvals.
5. Open your Day 2 folder and ask: *"Explain `add_ticket` line by line for a beginner."*
6. Ask: *"Write `is_valid_priority(p)` that accepts low, medium, high, with three pytest tests. Show a plan first."*
7. Read the diff. Run the tests. Accept only code you can explain.

The cells below practise the skills you need around Codex: reviewing diffs, scanning for risky code and estimating tokens.

## 8. NEW (v4): SmartDesk classes (step 2)

In [ ]:
class TicketNotFound(Exception):
    pass

class Customer:
    def __init__(self, name, plan="standard"):
        self.name, self.plan = name, plan

class Agent:
    def __init__(self, name, team="general"):
        self.name, self.team = name, team

class Ticket:
    PRIORITIES = ("low", "medium", "high")
    _next_id = 1

    def __init__(self, customer, issue, priority="medium"):
        if priority not in self.PRIORITIES:
            raise ValueError(f"priority must be one of {self.PRIORITIES}")
        self.id = Ticket._next_id
        Ticket._next_id += 1
        self.customer, self.issue, self.priority = customer, issue, priority
        self.status, self.agent = "open", None

    def assign(self, agent):
        self.agent = agent

    def close(self):
        self.status = "closed"

    def __str__(self):
        return f"#{self.id} [{self.priority}] {self.customer.name}: {self.issue} ({self.status})"

def find_ticket(tickets, ticket_id):
    for t in tickets:
        if t.id == ticket_id:
            return t
    raise TicketNotFound(f"No ticket {ticket_id}")

tickets = [Ticket(Customer("Mei"), "No signal at home", "high"),
           Ticket(Customer("Ravi", "premium"), "Charged twice this month"),
           Ticket(Customer("Aiko"), "Cannot log in to the app", "low")]
tickets[0].assign(Agent("Kenji", "network"))
for t in tickets:
    print(t)
try:
    find_ticket(tickets, 99)
except TicketNotFound as e:
    print("Error:", e)

## 9. NEW (v4): Review the diff
Imagine Codex proposed this `search_tickets`. It runs. Is it right? We show the diff, then run tests that a careful reviewer would write.

In [ ]:
import difflib
before = """def search_tickets(tickets, text):
    return []
"""
proposed = """def search_tickets(tickets, text):
    results = []
    for t in tickets:
        if text in t.customer.name:
            results.append(t)
    return results
"""
print("".join(difflib.unified_diff(before.splitlines(True), proposed.splitlines(True),
                                   fromfile="before.py", tofile="codex_proposal.py")))

ns = {}
exec(proposed, ns)        # load the proposal so we can test it (never do this with untrusted input)
search = ns["search_tickets"]
checks = {
    "finds Mei":            len(search(tickets, "Mei")) == 1,
    "case insensitive":     len(search(tickets, "mei")) == 1,
    "searches the issue":   len(search(tickets, "signal")) == 1,
    "no match is empty":    search(tickets, "zzz") == [],
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", name)

**Your fix** (this is what you would ask Codex for, then review again):

In [ ]:
def search_tickets(tickets, text):
    """Case insensitive search on customer name and issue."""
    q = text.lower().strip()
    return [t for t in tickets if q in t.customer.name.lower() or q in t.issue.lower()]

assert len(search_tickets(tickets, "mei")) == 1
assert len(search_tickets(tickets, "SIGNAL")) == 1
assert search_tickets(tickets, "zzz") == []
print("3 passed")

## 10. NEW (v4): Structured prompt template
Describe, plan, small steps, test. Fill it in and paste it into Codex.

In [ ]:
def codex_prompt(goal, files, rules, tests):
    return f"""Goal: {goal}
Files you may change: {", ".join(files)}
Rules: {"; ".join(rules)}
Tests first: {"; ".join(tests)}
Step 1: show me a short plan and wait for my approval.
Step 2: make the smallest change, then run the tests and show the diff."""

print(codex_prompt(
    goal="Add search_tickets(tickets, text) to SmartDesk",
    files=["smartdesk/models.py", "tests/test_search.py"],
    rules=["case insensitive", "match customer name or issue", "no new dependencies", "no eval or exec"],
    tests=["finds by name", "finds by issue", "empty list when no match"]))

## 11. NEW (v4): AGENTS.md basics
A starter file for your SmartDesk repo (template: `shared/templates/AGENTS_learner_template.md`).

In [ ]:
%%writefile AGENTS.md
# AGENTS.md: SmartDesk (learner repo)

## Commands
- Run tests: pytest -q
- Run the app: python smartdesk.py

## Style
- Python 3.12, type hints, short functions with docstrings.
- British spelling in user messages.

## Do not touch
- .env, .venv/, data/

## Rules
- Never put secrets, API keys or customer data in prompts or code.
- Add or update a test for every change.
- No new dependencies without asking; pin versions in requirements.txt.

# Session 4: NEW (v4): Using Codex responsibly
## 12. Security of AI-generated code
Three functions below look like they came from an AI tool. A tiny scanner flags risky patterns. Real tools (bandit, pip-audit, secret scanning) come on Day 7 and Day 20.

In [ ]:
import re
ai_code = {
    "calc.py":  'def calc(user_text):\n    return eval(user_text)\n',
    "client.py": 'API_KEY = "sk-live-not-a-real-key"\nheaders = {"Authorization": API_KEY}\n',
    "repo.py":  'def find(cur, name):\n    cur.execute(f"SELECT * FROM tickets WHERE customer = \'{name}\'")\n',
}
RULES = {
    r"\beval\(|\bexec\(": "code injection: never eval user input",
    r"shell\s*=\s*True": "shell injection: pass a list of arguments",
    r"(API_KEY|SECRET|PASSWORD)\s*=\s*[\"']": "secret in code: read it from the environment",
    r"execute\(f[\"']": "SQL injection: use query parameters",
}
for fname, src in ai_code.items():
    for pattern, why in RULES.items():
        if re.search(pattern, src):
            print(f"{fname:<10} {why}")

### TODO 3: Rewrite the three functions safely and re-run the scanner until it prints nothing.

In [ ]:
# TODO: write your code here

## 13. NEW (v4): Which model, which effort?
From `docs/MODEL_GUIDE.md`. Names checked 21 Oct 2026; confirm against the approved model list before class.

In [ ]:
GUIDE = [   # (keywords, model, effort)
    (("explain", "docstring", "rename", "small"), "lighter model (for example GPT-6 Luna, if approved)", "low"),
    (("debug", "migrate", "design", "large"),     "GPT-6.1 Sol at high effort (GPT-6 Astra only with trainer approval)", "high"),
    (("security", "patch", "vulnerab"),           "GPT-6.1 Sol, human verifies every fix", "medium"),
]
def pick(task):
    t = task.lower()
    for words, model, effort in GUIDE:
        if any(w in t for w in words):
            return model, effort
    return "GPT-6.1 Sol (default Codex model)", "medium"

for task in ["Explain this function", "Add search with tests", "Debug a failing migration",
             "Review a security patch"]:
    model, effort = pick(task)
    print(f"{task:<28} -> {effort:<6} {model}")

## 14. NEW (v4): Token cost of Codex use
Codex reads files and writes answers in tokens. Fewer files in context means fewer tokens. Prices are **placeholders**.

In [ ]:
def estimate_tokens(text):
    try:
        import tiktoken
        return len(tiktoken.get_encoding("o200k_base").encode(text))
    except Exception:
        return max(1, round(len(text) / 4))        # about 4 characters per token

PRICE_PER_1M = {"input": 1.00, "output": 4.00}    # PLACEHOLDER prices, not real
REASONING = {"low": 0.5, "medium": 1.5, "high": 4.0}   # rough multiplier on output, for teaching only

def codex_task_cost(prompt, files_text, expected_output_tokens, effort="medium"):
    inp = estimate_tokens(prompt) + estimate_tokens(files_text)
    out = expected_output_tokens * (1 + REASONING[effort])
    cost = (inp * PRICE_PER_1M["input"] + out * PRICE_PER_1M["output"]) / 1_000_000
    return inp, int(out), cost

one_file = open("pricing.py").read() * 20       # stand-in for one source file
tasks = [("explain one function", one_file, 300, "low"),
         ("add search + tests", one_file * 3, 900, "medium"),
         ("refactor 6 files", one_file * 6, 2500, "high")]
daily_budget = 0.50                                # placeholder budget
total = 0
for name, files, out_t, effort in tasks:
    i, o, c = codex_task_cost("Please do: " + name, files, out_t, effort)
    total += c
    print(f"{name:<22} effort={effort:<6} in={i:>6} out={o:>6} cost={c:.4f} (placeholder)")
print(f"Total {total:.4f} of a daily budget of {daily_budget} (placeholder units)")

### Daily usage check
At the end of each day, open your usage page for the program key and note the tokens used. If one task used far more than you expected, find out why (too many files, high effort, a loop) before you continue tomorrow.

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)